In [1]:
# ==============================================================
# Cell 1 — Setup & Imports
# ==============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# File path — update as needed
DATA_PATH="/Users/sriharikodam/Documents/IsbAMPBA/CapstoneProject/ClientData/Data/synthetic_encounters.csv"

print("Libraries imported and configuration ready.")

Libraries imported and configuration ready.


In [2]:
# ==============================================================
# Cell 2 — Load Data
# ==============================================================

def load_encounter_data(path: str) -> pd.DataFrame:
    """
    Load the encounter dataset from CSV, infer dtypes, and basic cleaning.
    """
    df = pd.read_csv(path)
    print(f"Loaded dataset with shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    return df

dfEnc = load_encounter_data(DATA_PATH)
dfEnc.head()

Loaded dataset with shape: (200000, 17)
Columns: ['encounter_id', 'patient_id', 'provider_id', 'encounter_date', 'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost', 'cultural_background', 'primary_language', 'languages_spoken', 'cultural_competency_rating', 'cultural_match_score', 'language_match', 'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days']


,encounter_id,patient_id,provider_id,encounter_date,encounter_type,primary_diagnosis,length_of_stay,total_cost,cultural_background,primary_language,languages_spoken,cultural_competency_rating,cultural_match_score,language_match,patient_satisfaction,treatment_adherence,return_visit_30_days
0,ENC_0000001,PAT_052723,PROV_04581,2025-04-04 19:28:51.502864,Office Visit,COPD,2,1078,European American,English,English,3.37,0.837,True,4.53,1.000,0.144
1,ENC_0000002,PAT_046599,PROV_00846,2024-05-28 19:28:51.502879,Office Visit,Asthma,2,2943,Hispanic/Latino,English,English,3.50,0.850,True,4.81,0.886,0.123
2,ENC_0000003,PAT_016355,PROV_01047,2025-06-17 19:28:51.502882,Office Visit,Essential Hypertension,1,10799,European American,English,English,3.67,0.867,True,4.31,0.906,0.120
3,ENC_0000004,PAT_016754,PROV_04343,2024-05-15 19:28:51.502883,Telehealth,Arthritis,3,1673,European American,English,English,3.47,0.847,True,5.00,1.000,0.129
4,ENC_0000005,PAT_074168,PROV_01228,2025-03-10 19:28:51.502885,Office Visit,Asthma,1,2959,European American,English,English,3.77,0.877,True,5.00,0.930,0.132


In [13]:
# ============================================
# Aggregate ENCOUNTER-level df → PATIENT–PROVIDER level
# Rules:
# - Categorical columns  → most frequent (mode)
# - length_of_stay       → max
# - Ratings (e.g., patient_satisfaction, cultural_competency_rating) → max
# - Scores / proportions (e.g., cultural_match_score, treatment_adherence, return_visit_30_days, language_match) → mean
# - total_cost           → mean (change to sum/median if you prefer)
# - Also keep: encounters count, last encounter_date
# ============================================

import numpy as np
import pandas as pd

# 0) Ensure df has at least these columns (adjust if your schema differs)
#    ['encounter_id','patient_id','provider_id','encounter_date',
#     'encounter_type','primary_diagnosis','length_of_stay','total_cost',
#     'cultural_background','primary_language','languages_spoken',
#     'cultural_competency_rating','cultural_match_score','language_match',
#     'patient_satisfaction','treatment_adherence','return_visit_30_days']

# 1) Keys for aggregation (patient–provider pairs)
keys = ["patient_id", "provider_id"]

# 2) Make a working copy and coerce types where needed
d = dfEnc.copy()
d["encounter_date"] = pd.to_datetime(d["encounter_date"], errors="coerce")

for c in ["length_of_stay","total_cost","cultural_competency_rating","cultural_match_score",
          "language_match","patient_satisfaction","treatment_adherence","return_visit_30_days"]:
    if c in d.columns:
        d[c] = pd.to_numeric(d[c], errors="coerce")

# 3) Helpers for mode (most frequent) with safe fallback
_mode = lambda s: s.mode().iat[0] if not s.mode().empty else np.nan

# 4) Encounters count per patient–provider
agg_cnt = d.groupby(keys, as_index=False)["encounter_id"].count().rename(columns={"encounter_id":"encounters"})

# 5) Last encounter date (max)
agg_last_dt = d.groupby(keys, as_index=False)["encounter_date"].max().rename(columns={"encounter_date":"last_encounter_date"})

# 6) Max length_of_stay
agg_los_max = d.groupby(keys, as_index=False)["length_of_stay"].max().rename(columns={"length_of_stay":"length_of_stay"})

# 7) Mean total_cost
agg_cost_mean = d.groupby(keys, as_index=False)["total_cost"].mean().rename(columns={"total_cost":"total_cost"})

# 8) Ratings → max
agg_rating_ps = d.groupby(keys, as_index=False)["patient_satisfaction"].max().rename(columns={"patient_satisfaction":"patient_satisfaction"})
agg_rating_cc = d.groupby(keys, as_index=False)["cultural_competency_rating"].max().rename(columns={"cultural_competency_rating":"cultural_competency_rating"})

# 9) Scores / proportions → mean
agg_match_mean   = d.groupby(keys, as_index=False)["cultural_match_score"].mean().rename(columns={"cultural_match_score":"cultural_match_score"})
agg_adher_mean   = d.groupby(keys, as_index=False)["treatment_adherence"].mean().rename(columns={"treatment_adherence":"treatment_adherence"})
agg_return_mean  = d.groupby(keys, as_index=False)["return_visit_30_days"].mean().rename(columns={"return_visit_30_days":"return_visit_30_days"})
agg_langmatch_mean = d.groupby(keys, as_index=False)["language_match"].mean().rename(columns={"language_match":"language_match"})

# 10) Categorical → most frequent (mode)
agg_enc_type = d.groupby(keys, as_index=False)["encounter_type"].agg(_mode).rename(columns={"encounter_type":"encounter_type"})
agg_dx_mode  = d.groupby(keys, as_index=False)["primary_diagnosis"].agg(_mode).rename(columns={"primary_diagnosis":"primary_diagnosis"})
agg_cult_bg  = d.groupby(keys, as_index=False)["cultural_background"].agg(_mode).rename(columns={"cultural_background":"cultural_background"})
agg_lang_prim= d.groupby(keys, as_index=False)["primary_language"].agg(_mode).rename(columns={"primary_language":"primary_language"})
agg_lang_spk = d.groupby(keys, as_index=False)["languages_spoken"].agg(_mode).rename(columns={"languages_spoken":"languages_spoken"})

# 11) Merge all aggregates into a single patient–provider DataFrame
df = agg_cnt.merge(agg_last_dt, on=keys, how="left")\
            .merge(agg_los_max, on=keys, how="left")\
            .merge(agg_cost_mean, on=keys, how="left")\
            .merge(agg_rating_ps, on=keys, how="left")\
            .merge(agg_rating_cc, on=keys, how="left")\
            .merge(agg_match_mean, on=keys, how="left")\
            .merge(agg_adher_mean, on=keys, how="left")\
            .merge(agg_return_mean, on=keys, how="left")\
            .merge(agg_langmatch_mean, on=keys, how="left")\
            .merge(agg_enc_type, on=keys, how="left")\
            .merge(agg_dx_mode,  on=keys, how="left")\
            .merge(agg_cult_bg,  on=keys, how="left")\
            .merge(agg_lang_prim,on=keys, how="left")\
            .merge(agg_lang_spk, on=keys, how="left")

# 12) Optional: recent_days per pair (min days since = most recent) — if you want recency
# latest_date = d["encounter_date"].max()
# d["recency_days"] = (latest_date - d["encounter_date"]).dt.days
# agg_recency_min = d.groupby(keys, as_index=False)["recency_days"].min().rename(columns={"recency_days":"recent_days"})
# pp = pp.merge(agg_recency_min, on=keys, how="left")

# 13) Final check
print(df.shape)
df.head()

(199964, 17)


,patient_id,provider_id,encounters,last_encounter_date,length_of_stay,total_cost,patient_satisfaction,cultural_competency_rating,cultural_match_score,treatment_adherence,return_visit_30_days,language_match,encounter_type,primary_diagnosis,cultural_background,primary_language,languages_spoken
0,PAT_000001,PROV_04499,1,2025-03-17 19:28:51.654206,5,1651.0,4.51,3.37,0.837,0.912,0.161,1.0,Telehealth,Back Pain,European American,English,English
1,PAT_000001,PROV_04538,1,2024-06-30 19:28:51.662524,2,1387.0,4.36,3.89,0.889,0.946,0.125,1.0,Office Visit,Asthma,European American,English,English; Spanish
2,PAT_000002,PROV_02628,1,2025-02-21 19:28:51.706079,0,16076.0,4.66,4.59,0.959,0.955,0.201,1.0,Office Visit,Arthritis,Hispanic/Latino,English,English; Arabic
3,PAT_000003,PROV_03728,1,2023-12-04 19:28:51.533494,4,816.0,4.08,3.40,0.540,0.757,0.176,0.0,Emergency,Hyperlipidemia,Asian American,Chinese,English
4,PAT_000004,PROV_00139,1,2023-12-25 19:28:51.723617,1,32815.0,4.44,4.16,0.616,0.771,0.173,0.0,Office Visit,Hyperlipidemia,European American,Spanish,English; Vietnamese


In [4]:
df['encounters'].value_counts()

encounters
1    199928
2        36
Name: count, dtype: int64

In [9]:
df.columns=dfEnc.columns

In [14]:
# ==============================================================

# Explicit type mapping from your provided schema
ID_COLS = ["encounter_id", "patient_id", "provider_id"]
DATE_COL = "encounter_date"
CATEGORICAL_COLS = [
    "encounter_type", "primary_diagnosis", "cultural_background",
    "primary_language", "languages_spoken"
]
ORDINAL_COLS = ["cultural_competency_rating", "patient_satisfaction"]
NUMERIC_COLS = ["length_of_stay", "total_cost", "cultural_match_score"]
BINARY_COLS = ["language_match"]

TARGET = "patient_satisfaction"

# Basic check for missing columns
for col in ID_COLS + CATEGORICAL_COLS + ORDINAL_COLS + NUMERIC_COLS + BINARY_COLS + [DATE_COL]:
    if col not in df.columns:
        print(f"⚠️ Missing expected column: {col}")

⚠️ Missing expected column: encounter_id
⚠️ Missing expected column: encounter_date


In [15]:
df['patient_satisfaction'].fillna

<bound method NDFrame.fillna of 0         4.51
1         4.36
2         4.66
3         4.08
4         4.44
          ... 
199959    4.95
199960    4.61
199961    4.96
199962    5.00
199963    4.10
Name: patient_satisfaction, Length: 199964, dtype: float64>

In [16]:
# --- Minimal, line-by-line snippet (no functions) ---
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

# Assume your DataFrame is named `df` and has at least:
# ['patient_id', 'patient_satisfaction']  (and optionally 'gender')

# 1) Make sure the score is numeric
df['patient_satisfaction'] = pd.to_numeric(df['patient_satisfaction'], errors='coerce')

# 2) Compute DENSE RANK within each patient (higher satisfaction = rank 1)
df['dense_rank'] = (
    df.groupby('patient_id')['patient_satisfaction']
      .rank(method='dense', ascending=False)
      .astype(int)
)

# 3) Turn the rank into an ordered categorical label like r1, r2, r3, ...
df['dense_rank_cat'] = 'r' + df['dense_rank'].astype(str)

# 4) Build the category order explicitly (ensures OrdinalEncoder follows rank order)
rank_levels_sorted = sorted(df['dense_rank'].unique())               # e.g., [1,2,3,...]
rank_categories = [['r' + str(x) for x in rank_levels_sorted]]       # e.g., [['r1','r2','r3',...]]

# 5) Ordinal-encode the rank category to integers starting at 0 (r1->0, r2->1, ...)
enc_rank = OrdinalEncoder(categories=rank_categories)
df[['dense_rank_enc']] = enc_rank.fit_transform(df[['dense_rank_cat']]).astype(int)

### # 1) Make sure the score is numeric
df['cultural_competency_rating'] = pd.to_numeric(df['cultural_competency_rating'], errors='coerce')

# 2) Compute DENSE RANK within each patient (higher satisfaction = rank 1)
df['dense_rank_culturalComp'] = (
    df.groupby('patient_id')['cultural_competency_rating']
      .rank(method='dense', ascending=False)
      .astype(int)
)

# 3) Turn the rank into an ordered categorical label like r1, r2, r3, ...
df['dense_rank_cat_culturalComp'] = 'r' + df['dense_rank_culturalComp'].astype(str)

# 4) Build the category order explicitly (ensures OrdinalEncoder follows rank order)
rank_levels_sorted = sorted(df['dense_rank_culturalComp'].unique())               # e.g., [1,2,3,...]
rank_categories = [['r' + str(x) for x in rank_levels_sorted]]       # e.g., [['r1','r2','r3',...]]

# 5) Ordinal-encode the rank category to integers starting at 0 (r1->0, r2->1, ...)
enc_rank = OrdinalEncoder(categories=rank_categories)
df['dense_rank_enc_culturalComp'] = enc_rank.fit_transform(df[['dense_rank_cat_culturalComp']]).astype(int)
######

# (Optional) If you also want to encode gender as ordinal: female < male
# Normalize to lowercase to avoid mismatches
if 'gender' in df.columns:
    df['gender'] = df['gender'].astype(str).str.strip().str.lower()

    # 2) Apply OneHotEncoder (dense output for simplicity)
    ohe_gender = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    # 3) Fit and transform the gender column
    gender_encoded = ohe_gender.fit_transform(enc[['gender']])

    # 4) Get the column names like gender_female, gender_male, etc.
    gender_ohe_cols = [f"gender_{cat}" for cat in ohe_gender.categories_[0]]

    # 5) Add back to the DataFrame
    df[gender_ohe_cols] = gender_encoded

    # 6) (Optional) Drop the original gender column if not needed
    df = df.drop(columns=['gender'])


# 6) (Optional) Drop helper columns if you only want the encoded result
# df = df.drop(columns=['dense_rank', 'dense_rank_cat'])

# 7) Quick sanity check
print(df[['patient_id', 'patient_satisfaction', 'dense_rank', 'dense_rank_cat', 'dense_rank_enc','dense_rank_enc_culturalComp']].head())
# If gender used: print(df[['gender','gender_enc']].drop_duplicates().head())

   patient_id  patient_satisfaction  dense_rank dense_rank_cat  \
0  PAT_000001                  4.51           1             r1   
1  PAT_000001                  4.36           2             r2   
2  PAT_000002                  4.66           1             r1   
3  PAT_000003                  4.08           1             r1   
4  PAT_000004                  4.44           1             r1   

   dense_rank_enc  dense_rank_enc_culturalComp  
0               0                            1  
1               1                            0  
2               0                            0  
3               0                            0  
4               0                            0  


In [17]:
df.columns

Index(['patient_id', 'provider_id', 'encounters', 'last_encounter_date',
       'length_of_stay', 'total_cost', 'patient_satisfaction',
       'cultural_competency_rating', 'cultural_match_score',
       'treatment_adherence', 'return_visit_30_days', 'language_match',
       'encounter_type', 'primary_diagnosis', 'cultural_background',
       'primary_language', 'languages_spoken', 'dense_rank', 'dense_rank_cat',
       'dense_rank_enc', 'dense_rank_culturalComp',
       'dense_rank_cat_culturalComp', 'dense_rank_enc_culturalComp'],
      dtype='object')

In [18]:
import pandas as pd
import numpy as np

# If not already defined:
DATE_COL = "encounter_date"
ORDINAL_COLS = ["cultural_competency_rating", "patient_satisfaction"]  # adjust if needed

# # 1) Ensure datetime
# df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

# # 2) Recency in days (days since most recent encounter in the data)
# _max_date = df[DATE_COL].max()
# df["recency_days"] = (_max_date - df[DATE_COL]).dt.days

# 3) Cultural match score → percentage (if present; tolerant to existing 0–100 or 0–1)
if "cultural_match_score" in df.columns:
    df["cultural_match_score"] = pd.to_numeric(df["cultural_match_score"], errors="coerce")
    # If values look like proportions (<=1.5), convert to percent; else keep as-is
    _cms_max = df["cultural_match_score"].dropna().max()
    df["cultural_match_score"] = np.where(
        (_cms_max is not np.nan) & (df["cultural_match_score"] <= 1.5),
        df["cultural_match_score"] * 100.0,
        df["cultural_match_score"]
    )

# 5) Binary column cleanup (cast to 0/1 int if present)
if "language_match" in df.columns:
    df["language_match"] = pd.to_numeric(df["language_match"], errors="coerce").fillna(0)
    # Map common textual/bool variants to d/1 then cast
    df["language_match"] = df["language_match"].replace({True:1, False:0})
    df["language_match"] = df["language_match"].astype(int)

# 6) Quick peek
df.head()

,patient_id,provider_id,encounters,last_encounter_date,length_of_stay,total_cost,patient_satisfaction,cultural_competency_rating,cultural_match_score,treatment_adherence,...,primary_diagnosis,cultural_background,primary_language,languages_spoken,dense_rank,dense_rank_cat,dense_rank_enc,dense_rank_culturalComp,dense_rank_cat_culturalComp,dense_rank_enc_culturalComp
0,PAT_000001,PROV_04499,1,2025-03-17 19:28:51.654206,5,1651.0,4.51,3.37,83.7,0.912,...,Back Pain,European American,English,English,1,r1,0,2,r2,1
1,PAT_000001,PROV_04538,1,2024-06-30 19:28:51.662524,2,1387.0,4.36,3.89,88.9,0.946,...,Asthma,European American,English,English; Spanish,2,r2,1,1,r1,0
2,PAT_000002,PROV_02628,1,2025-02-21 19:28:51.706079,0,16076.0,4.66,4.59,95.9,0.955,...,Arthritis,Hispanic/Latino,English,English; Arabic,1,r1,0,1,r1,0
3,PAT_000003,PROV_03728,1,2023-12-04 19:28:51.533494,4,816.0,4.08,3.40,54.0,0.757,...,Hyperlipidemia,Asian American,Chinese,English,1,r1,0,1,r1,0
4,PAT_000004,PROV_00139,1,2023-12-25 19:28:51.723617,1,32815.0,4.44,4.16,61.6,0.771,...,Hyperlipidemia,European American,Spanish,English; Vietnamese,1,r1,0,1,r1,0


In [19]:
# --- Simple, line-by-line feature engineering for categorical OHE ---

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# Assumes your DataFrame is named `enc`
# Columns used: primary_diagnosis, encounter_type, cultural_background, languages_spoken, primary_language

# 1) Rename primary_language -> provider_language
df = df.rename(columns={"primary_language": "provider_language"})

# 2) Normalize text columns (strip + lowercase) for stable categories
for _c in ["primary_diagnosis", "encounter_type", "cultural_background", "provider_language", "languages_spoken"]:
    if _c in df.columns:
        df[_c] = df[_c].astype(str).str.strip().str.lower().replace({"nan": np.nan, "": np.nan})

# 3) Split languages_spoken into language_1 and language_2 (split by ';')
if "languages_spoken" in df.columns:
    _langs = df["languages_spoken"].fillna("").astype(str).str.split(";")
    df["language_1"] = _langs.apply(lambda x: x[0].strip().lower() if len(x) >= 1 and x[0].strip() != "" else np.nan)
    df["language_2"] = _langs.apply(lambda x: x[1].strip().lower() if len(x) >= 2 and x[1].strip() != "" else "none")
else:
    df["language_1"] = np.nan
    df["language_2"] = "none"

# 4) Choose categorical columns to one-hot encode
cat_cols = [
    "primary_diagnosis",
    "encounter_type",
    "cultural_background",
    "language_1",
    "language_2",
    "provider_language"
]

# 5) Impute missing with most frequent (mode) per column (simple, inline)
for _c in cat_cols:
    if _c in df.columns:
        _series = df[_c]
        if _series.dropna().empty:
            _mode = "unknown"
        else:
            _mode = _series.mode(dropna=True).iloc[0]
        df[_c] = _series.fillna(_mode)
    else:
        df[_c] = "unknown"

# 6) OneHotEncode the selected categorical columns (dense output for simplicity)
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)  # sklearn >= 1.2
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)         # sklearn < 1.2

X_cat = ohe.fit_transform(df[cat_cols])
ohe_cols = ohe.get_feature_names_out(cat_cols)

# 7) Create a DataFrame for the encoded features and align indices
df_ohe = pd.DataFrame(X_cat, columns=ohe_cols, index=df.index)

# 8) Concatenate back and (optionally) drop original categorical columns
df = pd.concat([df.drop(columns=["languages_spoken"] + cat_cols, errors="ignore"), df_ohe], axis=1)

In [20]:
import numpy as np
import pandas as pd

# Assume your dataframe name is enc

# 1) Convert total_cost to numeric (in case it has strings or missing values)
df["total_cost"] = pd.to_numeric(df["total_cost"], errors="coerce")

# 2) Apply log transformation: log1p handles zeros safely (log(1 + x))
df["total_cost_log"] = np.log1p(df["total_cost"])

# 3) Convert length_of_stay to numeric
df["length_of_stay"] = pd.to_numeric(df["length_of_stay"], errors="coerce")

# 4) Define bin edges and labels
bin_edges = [-np.inf, 1, 3, 7, 14, np.inf]
bin_labels = ["<=1d", "2-3d", "4-7d", "8-14d", ">14d"]

# 5) Create binned variable using pd.cut
df["length_of_stay_bin"] = pd.cut(df["length_of_stay"], bins=bin_edges, labels=bin_labels)

df["length_of_stay_bin"] = df["length_of_stay_bin"].astype(str).str.strip().replace({"nan": np.nan})

df["length_of_stay_bin"] = df["length_of_stay_bin"].fillna("unknown")

# 3) Initialize OneHotEncoder (dense output for simplicity)
try:
    ohe_los = OneHotEncoder(handle_unknown="ignore", sparse_output=False)  # sklearn ≥1.2
except TypeError:
    ohe_los = OneHotEncoder(handle_unknown="ignore", sparse=False)         # sklearn <1.2

# 4) Fit & transform the length_of_stay_bin column
los_encoded = ohe_los.fit_transform(df[["length_of_stay_bin"]])

# 5) Get encoded column names
los_ohe_cols = [f"length_of_stay_bin_{cat}" for cat in ohe_los.categories_[0]]

# 6) Create a DataFrame for encoded columns
df_los_ohe = pd.DataFrame(los_encoded, columns=los_ohe_cols, index=df.index)

# 7) Add the new columns back to the main DataFrame
df = pd.concat([df.drop(columns=["length_of_stay_bin"], errors="ignore"), df_los_ohe], axis=1)

# 6) Quick sanity check
print(df[["total_cost", "total_cost_log", "length_of_stay", "length_of_stay_bin_2-3d", "length_of_stay_bin_4-7d",'length_of_stay_bin_8-14d', 'length_of_stay_bin_<=1d']].head())

   total_cost  total_cost_log  length_of_stay  length_of_stay_bin_2-3d  \
0      1651.0        7.409742               5                      0.0   
1      1387.0        7.235619               2                      1.0   
2     16076.0        9.685145               0                      0.0   
3       816.0        6.705639               4                      0.0   
4     32815.0       10.398671               1                      0.0   

   length_of_stay_bin_4-7d  length_of_stay_bin_8-14d  length_of_stay_bin_<=1d  
0                      1.0                       0.0                      0.0  
1                      0.0                       0.0                      0.0  
2                      0.0                       0.0                      1.0  
3                      1.0                       0.0                      0.0  
4                      0.0                       0.0                      1.0  


['encounter_id', 'patient_id', 'provider_id', 'encounter_date', 'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost', 'cultural_background', 'primary_language', 'languages_spoken', 'cultural_competency_rating', 'cultural_match_score', 'language_match', 'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days']

In [42]:
df.columns

Index(['encounter_id', 'patient_id', 'provider_id', 'encounter_date',
       'length_of_stay', 'total_cost', 'cultural_competency_rating',
       'cultural_match_score', 'language_match', 'patient_satisfaction',
       'treatment_adherence', 'return_visit_30_days', 'dense_rank',
       'dense_rank_cat', 'dense_rank_enc', 'dense_rank_culturalComp',
       'dense_rank_cat_culturalComp', 'dense_rank_enc_culturalComp',
       'primary_diagnosis_anxiety', 'primary_diagnosis_arthritis',
       'primary_diagnosis_asthma', 'primary_diagnosis_back pain',
       'primary_diagnosis_copd', 'primary_diagnosis_coronary artery disease',
       'primary_diagnosis_depression',
       'primary_diagnosis_essential hypertension',
       'primary_diagnosis_hyperlipidemia', 'primary_diagnosis_migraine',
       'primary_diagnosis_obesity', 'primary_diagnosis_type 2 diabetes',
       'encounter_type_emergency', 'encounter_type_inpatient',
       'encounter_type_office visit', 'encounter_type_telehealth',
    

In [21]:
df.iloc[0]

patient_id                                                   PAT_000001
provider_id                                                  PROV_04499
encounters                                                            1
last_encounter_date                          2025-03-17 19:28:51.654206
length_of_stay                                                        5
total_cost                                                       1651.0
patient_satisfaction                                               4.51
cultural_competency_rating                                         3.37
cultural_match_score                                               83.7
treatment_adherence                                               0.912
return_visit_30_days                                              0.161
language_match                                                        1
dense_rank                                                            1
dense_rank_cat                                                  

In [22]:
df['dense_rank_enc'].value_counts()

dense_rank_enc
0    98805
1    57430
2    28440
3    10838
4     3372
5      850
6      177
7       37
8       10
9        5
Name: count, dtype: int64

In [23]:
df.columns

Index(['patient_id', 'provider_id', 'encounters', 'last_encounter_date',
       'length_of_stay', 'total_cost', 'patient_satisfaction',
       'cultural_competency_rating', 'cultural_match_score',
       'treatment_adherence', 'return_visit_30_days', 'language_match',
       'dense_rank', 'dense_rank_cat', 'dense_rank_enc',
       'dense_rank_culturalComp', 'dense_rank_cat_culturalComp',
       'dense_rank_enc_culturalComp', 'primary_diagnosis_anxiety',
       'primary_diagnosis_arthritis', 'primary_diagnosis_asthma',
       'primary_diagnosis_back pain', 'primary_diagnosis_copd',
       'primary_diagnosis_coronary artery disease',
       'primary_diagnosis_depression',
       'primary_diagnosis_essential hypertension',
       'primary_diagnosis_hyperlipidemia', 'primary_diagnosis_migraine',
       'primary_diagnosis_obesity', 'primary_diagnosis_type 2 diabetes',
       'encounter_type_emergency', 'encounter_type_inpatient',
       'encounter_type_office visit', 'encounter_type_telehea

In [24]:
featurereqColumns= ['dense_rank_enc','dense_rank_culturalComp', 'dense_rank_cat_culturalComp',
       'dense_rank_enc_culturalComp', 'primary_diagnosis_anxiety',
       'primary_diagnosis_arthritis', 'primary_diagnosis_asthma',
       'primary_diagnosis_back pain', 'primary_diagnosis_copd',
       'primary_diagnosis_coronary artery disease',
       'primary_diagnosis_depression',
       'primary_diagnosis_essential hypertension',
       'primary_diagnosis_hyperlipidemia', 'primary_diagnosis_migraine',
       'primary_diagnosis_obesity', 'primary_diagnosis_type 2 diabetes',
       'encounter_type_emergency', 'encounter_type_inpatient',
       'encounter_type_office visit', 'encounter_type_telehealth',
       'cultural_background_african american',
       'cultural_background_asian american',
       'cultural_background_european american',
       'cultural_background_hispanic/latino',
       'cultural_background_native american',
       'cultural_background_other/mixed', 'language_1_english',
       'language_2_arabic', 'language_2_chinese', 'language_2_korean',
       'language_2_none', 'language_2_spanish', 'language_2_vietnamese',
       'provider_language_arabic', 'provider_language_chinese',
       'provider_language_english', 'provider_language_korean',
       'provider_language_portuguese', 'provider_language_russian',
       'provider_language_spanish', 'provider_language_vietnamese',
       'total_cost_log', 'length_of_stay_bin_2-3d', 'length_of_stay_bin_4-7d',
       'length_of_stay_bin_8-14d', 'length_of_stay_bin_<=1d']
feature_cols = [c for c in featurereqColumns if c != "dense_rank_enc"]

In [25]:
train_merged = df[feature_cols]
X = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype("float32").values
encounter_max_rank = df.groupby("patient_id")["dense_rank_enc"].transform("max")
df["label_int"] = np.where(df["dense_rank_enc"] == encounter_max_rank, 1, 0)
groups_train = df.groupby("patient_id").size().values
y = df["label_int"].astype(int).values


['dense_rank_enc','dense_rank_enc_culturalComp',
       'primary_diagnosis_anxiety', 'primary_diagnosis_arthritis',
       'primary_diagnosis_asthma', 'primary_diagnosis_back pain',
       'primary_diagnosis_copd', 'primary_diagnosis_coronary artery disease',
       'primary_diagnosis_depression',
       'primary_diagnosis_essential hypertension',
       'primary_diagnosis_hyperlipidemia', 'primary_diagnosis_migraine',
       'primary_diagnosis_obesity', 'primary_diagnosis_type 2 diabetes',
       'encounter_type_emergency', 'encounter_type_inpatient',
       'encounter_type_office visit', 'encounter_type_telehealth',
       'cultural_background_african american',
       'cultural_background_asian american',
       'cultural_background_european american',
       'cultural_background_hispanic/latino',
       'cultural_background_native american',
       'cultural_background_other/mixed', 'language_1_english',
       'language_2_arabic', 'language_2_chinese', 'language_2_korean',
       'language_2_none', 'language_2_spanish', 'language_2_vietnamese',
       'provider_language_arabic', 'provider_language_chinese',
       'provider_language_english', 'provider_language_korean',
       'provider_language_portuguese', 'provider_language_russian',
       'provider_language_spanish', 'provider_language_vietnamese',
       'total_cost_log', 'length_of_stay_bin_2-3d', 'length_of_stay_bin_4-7d',
       'length_of_stay_bin_8-14d', 'length_of_stay_bin_<=1d',
       'length_of_stay_bin_2-3d', 'length_of_stay_bin_4-7d',
       'length_of_stay_bin_8-14d', 'length_of_stay_bin_<=1d',
       'length_of_stay_bin_2-3d', 'length_of_stay_bin_4-7d',
       'length_of_stay_bin_8-14d', 'length_of_stay_bin_<=1d',
       'length_of_stay_bin_2-3d', 'length_of_stay_bin_4-7d',
       'length_of_stay_bin_8-14d', 'length_of_stay_bin_<=1d',
       'length_of_stay_bin_2-3d', 'length_of_stay_bin_4-7d',
       'length_of_stay_bin_8-14d', 'length_of_stay_bin_<=1d']

In [66]:
groups_train.shape

(86478,)

In [57]:
# Count of zeros in y_train
print("Number of zeros in y_train:", (y_train == 0).sum())

# Optional: percentage of zeros
print("Percentage of zeros:", ((y_train == 0).sum() / len(y_train)) * 100, "%")

Number of zeros in y_train: 109837
Percentage of zeros: 54.9185 %


In [67]:
# Count of zeros in y_train
print("Number of zeros in y_train:", (groups_train == 0).sum())

# Optional: percentage of zeros
print("Percentage of zeros:", ((groups_train == 0).sum() / len(groups_train)) * 100, "%")

Number of zeros in y_train: 0
Percentage of zeros: 0.0 %


In [28]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df["patient_id"]))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train = df.iloc[train_idx].groupby("patient_id").size().values

In [29]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRanker
from xgboost import XGBRanker

# 14) Fit LGBMRanker
lgb = LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)
lgb.fit(X_train, y_train, group=groups_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005082 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 358
[LightGBM] [Info] Number of data points in the train set: 159861, number of used features: 43


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.05
,n_estimators,400
,subsample_for_bin,200000
,objective,'lambdarank'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [34]:
xgb = XGBRanker(
    objective="rank:ndcg",
    n_estimators=400,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method="hist",
    eval_metric="ndcg",
    random_state=42
)
xgb.fit(X_train, y_train, group=groups_train)

,objective,'rank:ndcg'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.9
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'ndcg'


In [35]:
scores_lgb = lgb.predict(X_test).astype(float)
# scores_xgb = xgb.predict(X_test).astype(float)

# 19) Rank providers per encounter — LGBM
N = 4  # top-N providers per encounter
rows_lgb = []
start = 0
for eid, g in df.iloc[test_idx].groupby("patient_id", sort=False):
    n = len(g)
    stop = start + n
    sc = scores_lgb[start:stop]
    order = np.argsort(-sc)[:N]
    top = g.iloc[order][["patient_id","provider_id"]].copy()
    top["rank"]  = np.arange(1, len(top)+1)
    top["score"] = sc[order]
    rows_lgb.append(top)
    start = stop
rankings_lgb = pd.concat(rows_lgb, axis=0).reset_index(drop=True)

In [36]:
rankings_lgb

,patient_id,provider_id,rank,score
0,PAT_000029,PROV_02046,1,0.258489
1,PAT_000029,PROV_00083,2,0.213507
2,PAT_000029,PROV_02178,3,0.058965
3,PAT_000029,PROV_03981,4,-0.250992
4,PAT_000040,PROV_01982,1,0.145463
...,...,...,...,...
38518,PAT_099978,PROV_03609,1,0.031163
38519,PAT_099978,PROV_00091,2,-0.030087
38520,PAT_099994,PROV_02536,1,-0.056474
38521,PAT_099995,PROV_01840,1,0.013021


In [80]:
dfEnc.groupby(keys)['encounter_id'].count().sort_values(ascending=False)

patient_id  provider_id
PAT_012100  PROV_04755     2
PAT_091773  PROV_02755     2
PAT_010749  PROV_02589     2
PAT_030138  PROV_02624     2
PAT_075193  PROV_00838     2
                          ..
PAT_033449  PROV_02332     1
            PROV_02660     1
            PROV_04199     1
            PROV_04224     1
PAT_099999  PROV_02146     1
Name: encounter_id, Length: 199964, dtype: int64

In [37]:
# ================================================
# Line-by-line evaluation for LGBM/XGB rankers
# Metrics: NDCG@{3,5,7}, Precision@{3,5,7}, Recall@{3,5,7}, MAP@{3,5,7}
# Assumptions:
#   - df_test has columns: ['patient_id','provider_id','label_int'] aligned to X_test row order
#   - lgb (LGBMRanker) and xgb (XGBRanker) are already fitted
#   - X_test is the feature matrix for df_test in the same row order
# ================================================

import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score

# 0) Predict scores
scores_lgb = lgb.predict(X_test).astype(float)
scores_xgb = xgb.predict(X_test).astype(float)

# 1) Keep only queries (patients) with at least 2 candidates (else NDCG is undefined)
#    Also ensure df_test is sorted/grouped exactly like X_test rows
df_eval = df.iloc[test_idx].copy()

# 2) Build group sizes in current order
group_sizes = df_eval.groupby("patient_id").size().values

# 3) Choose cutoffs
Ks = [3, 5, 7]

# 4) Prepare accumulators for both models
acc_lgb = {f"ndcg@{k}":0.0 for k in Ks} | {f"precision@{k}":0.0 for k in Ks} | {f"recall@{k}":0.0 for k in Ks} | {f"map@{k}":0.0 for k in Ks}
acc_xgb = {f"ndcg@{k}":0.0 for k in Ks} | {f"precision@{k}":0.0 for k in Ks} | {f"recall@{k}":0.0 for k in Ks} | {f"map@{k}":0.0 for k in Ks}

# 5) Iterate groups and accumulate metrics (skip degenerate groups)
start = 0
eval_groups_lgb = 0
eval_groups_xgb = 0
skipped_groups = 0

for gsize in group_sizes:
    stop = start + gsize
    y = df_eval.iloc[start:stop]["label_int"].to_numpy()
    s_lgb = scores_lgb[start:stop]
    s_xgb = scores_xgb[start:stop]

    # Skip groups with <2 candidates or all-equal labels (no ranking signal)
    if gsize < 2 or np.all(y == y[0]):
        skipped_groups += 1
        start = stop
        continue

    # --- Graded NDCG (uses label_int directly) ---
    y_row = y.reshape(1, -1)
    lgb_row = s_lgb.reshape(1, -1)
    xgb_row = s_xgb.reshape(1, -1)
    for k in Ks:
        acc_lgb[f"ndcg@{k}"] += float(ndcg_score(y_row, lgb_row, k=k))
        acc_xgb[f"ndcg@{k}"] += float(ndcg_score(y_row, xgb_row, k=k))

    # --- Binary relevance for P/R/MAP: treat max label as relevant ---
    y_bin = (y == y.max()).astype(int)

    # LGB: compute Precision/Recall/MAP@k
    order_lgb = np.argsort(-s_lgb)
    for k in Ks:
        top_idx = order_lgb[:k]
        hits = y_bin[top_idx]
        precision = float(hits.mean())
        recall = float(hits.sum()) / max(1, int(y_bin.sum()))
        # MAP@k
        cum = 0; ap = 0.0; denom = min(k, max(1, int(y_bin.sum())))
        for rank_pos, idx in enumerate(top_idx, 1):
            if y_bin[idx] == 1:
                cum += 1
                ap += cum / rank_pos
        ap /= denom
        acc_lgb[f"precision@{k}"] += precision
        acc_lgb[f"recall@{k}"]    += recall
        acc_lgb[f"map@{k}"]       += ap
    eval_groups_lgb += 1

    # XGB: compute Precision/Recall/MAP@k
    order_xgb = np.argsort(-s_xgb)
    for k in Ks:
        top_idx = order_xgb[:k]
        hits = y_bin[top_idx]
        precision = float(hits.mean())
        recall = float(hits.sum()) / max(1, int(y_bin.sum()))
        # MAP@k
        cum = 0; ap = 0.0; denom = min(k, max(1, int(y_bin.sum())))
        for rank_pos, idx in enumerate(top_idx, 1):
            if y_bin[idx] == 1:
                cum += 1
                ap += cum / rank_pos
        ap /= denom
        acc_xgb[f"precision@{k}"] += precision
        acc_xgb[f"recall@{k}"]    += recall
        acc_xgb[f"map@{k}"]       += ap
    eval_groups_xgb += 1

    start = stop

# 6) Average over evaluated groups
for d, n in [(acc_lgb, eval_groups_lgb), (acc_xgb, eval_groups_xgb)]:
    if n == 0:
        continue
    for k in list(d.keys()):
        d[k] = d[k] / n

# 7) Build results table
rows = []
rows.append({"Model":"LGBMRanker", **acc_lgb, "evaluated_groups": eval_groups_lgb, "skipped_groups": skipped_groups})
rows.append({"Model":"XGBRanker", **acc_xgb, "evaluated_groups": eval_groups_xgb, "skipped_groups": skipped_groups})
results_df = pd.DataFrame(rows)

# 8) Show results
print(results_df)

        Model    ndcg@3    ndcg@5    ndcg@7  precision@3  precision@5  \
0  LGBMRanker  0.704604  0.734482  0.737239     0.382407     0.381456   
1   XGBRanker  0.708100  0.738393  0.740480     0.382701     0.381879   

   precision@7  recall@3  recall@5  recall@7     map@3     map@5     map@7  \
0     0.381825  0.921106  0.991899  0.999648  0.629025  0.646155  0.647469   
1     0.381837  0.922074  0.993836  0.999692  0.633361  0.650762  0.651757   

   evaluated_groups  skipped_groups  
0             11357            5939  
1             11357            5939  
